<a href="https://colab.research.google.com/github/syntaxJD/lab-3/blob/main/lab_3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [4]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
RANDOM_STATE = 42
torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)

print("Running on:", DEVICE)

Running on: cpu


In [ ]:
# --- Generate the toy data ---
x = np.linspace(-3, 3, 100)
y = np.sin(x) + np.random.normal(0, 0.1, size=x.shape)

plt.figure()
plt.scatter(x, y, s=10)
plt.title("Toy data: y = sin(x) + noise")
plt.xlabel("x")
plt.ylabel("y")
plt.show()

In [ ]:
w = np.random.randn() * 0.1
b = np.random.randn() * 0.1

In [ ]:
lr = 0.05
steps = 200
losses = []

for step in range(steps):
    y_hat = w * x + b
    loss = np.mean((y_hat - y) ** 2)
    dw = np.mean(2 * (y_hat - y) * x)
    db = np.mean(2 * (y_hat - y))
    w -= lr * dw
    b -= lr * db
    losses.append(loss)

print(f"Final w={w:.3f}, b={b:.3f}, final loss={losses[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10, 4))

axes[0].plot(losses)
axes[0].set_title("Loss over training steps")
axes[0].set_xlabel("Step")
axes[0].set_ylabel("MSE Loss")

axes[1].scatter(x, y, s=10, label="data")
axes[1].plot(x, w * x + b, color="red", label="fitted line")
axes[1].set_title("Fit: single neuron")
axes[1].legend()

plt.show()

### Student Reasoning — The single neuron

**1. Derivation of dLoss/dw**

The model and loss are:

$$
\hat{y}_i = w x_i + b
$$

$$
L = \frac{1}{n} \sum_{i=1}^{n} (\hat{y}_i - y_i)^2 = \frac{1}{n} \sum_{i=1}^{n} (w x_i + b - y_i)^2
$$

To find $\frac{\partial L}{\partial w}$, apply the chain rule. Let $u_i = w x_i + b - y_i$, so $L = \frac{1}{n}\sum u_i^2$.

$$
\frac{\partial L}{\partial w} = \frac{1}{n} \sum_{i=1}^{n} \frac{\partial}{\partial w}\left(u_i^2\right)
= \frac{1}{n} \sum_{i=1}^{n} 2 u_i \cdot \frac{\partial u_i}{\partial w}
$$

Since $u_i = w x_i + b - y_i$, we have $\frac{\partial u_i}{\partial w} = x_i$. Substituting:

$$
\frac{\partial L}{\partial w} = \frac{1}{n} \sum_{i=1}^{n} 2(w x_i + b - y_i)\, x_i
$$

$$
\boxed{\frac{\partial L}{\partial w} = \frac{2}{n}\sum_{i=1}^{n} (\hat{y}_i - y_i)\, x_i}
$$

Similarly for $b$, since $\frac{\partial u_i}{\partial b} = 1$:

$$
\frac{\partial L}{\partial b} = \frac{2}{n}\sum_{i=1}^{n} (\hat{y}_i - y_i)
$$

**This matches the code exactly:**

```python
dw = np.mean(2 * (y_hat - y) * x)   # = (2/n) * sum((y_hat - y) * x)
db = np.mean(2 * (y_hat - y))       # = (2/n) * sum(y_hat - y)
```

`np.mean` computes $\frac{1}{n}\sum(\cdot)$, and multiplying by the `2 * (y_hat - y) * x` term inside gives exactly $\frac{2}{n}\sum(\hat{y}_i-y_i)x_i$ — identical to the derived formula.

---

**2. Underfitting, not overfitting**

The loss plateaus but the fit is still poor because a single neuron computes $\hat{y} = wx + b$, which is the equation of a **straight line**. No matter how long we train, or how small the loss gets relative to what a line can achieve, the *hypothesis space* of the model is restricted to lines. The function $y = \sin(x)$ is inherently curved, so there is no choice of $w$ and $b$ that can trace it — the best the model can do is find the single straight line that minimizes average squared error across the curve (roughly a line through the "middle" of the oscillation).

This is **underfitting**: the model's capacity, not the optimization process, is the bottleneck. Overfitting would instead look like the training loss continuing to drop while validation loss rises — i.e., the model has *enough* capacity to fit the data (even its noise) but generalizes poorly. Here we have the opposite problem: the model doesn't have enough capacity to represent the true function at all, regardless of how well-tuned the learning rate or how many steps we run.